# 4. Sampling Techniques

**Statistical Foundations for Data Science — Notebook 4 of 8**

You almost never have the whole population. You have a **sample**, and you want to say
something about the population it came from. That leap is only valid if the sample was
collected properly — no statistical technique can rescue a badly drawn sample.

> *"The most dangerous number in data science is one computed from a biased sample, because
> it looks exactly like a good number."*

### What you will learn

1. Population vs. sample; parameter vs. statistic
2. **Probability sampling:** simple random, systematic, stratified, cluster, multi-stage
3. **Non-probability sampling** and the biases it introduces
4. **Sampling distributions** and the **standard error**
5. The **Central Limit Theorem** — stated, demonstrated, and stress-tested
6. **Confidence intervals** — what they do and do not mean
7. **Bootstrap** resampling: confidence intervals without formulas
8. **Sample size** calculations
9. Train/test splitting and cross-validation as sampling problems

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=11)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

---
## 4.1 Population, sample, parameter, statistic

| | Population | Sample |
|---|---|---|
| Definition | Every unit you care about | The subset you actually measured |
| Size | $N$ | $n$ |
| Mean | $\mu$ (a **parameter**) | $\bar{x}$ (a **statistic**) |
| SD | $\sigma$ | $s$ |
| Proportion | $p$ | $\hat{p}$ |
| Known? | Almost never | Always |

**Parameters are fixed but unknown. Statistics are known but random.** Every statistic you
compute would come out slightly different with a different sample — and quantifying that
"slightly" is the entire subject of statistical inference.

Greek letters for parameters, Latin letters (or hats) for statistics. This convention is
universal; use it and your work becomes readable.

In [ ]:
# Build a synthetic "population" so we can cheat and know the true parameters.
N = 50_000
population = pd.DataFrame({
    "id": np.arange(N),
    "region": rng.choice(["North", "South", "East", "West"], size=N, p=[0.4, 0.3, 0.2, 0.1]),
})
# Income depends on region -> this is what makes stratification worthwhile later
region_effect = {"North": 0.35, "South": 0.0, "East": -0.15, "West": 0.55}
population["income"] = np.exp(
    10.6 + population["region"].map(region_effect) + rng.normal(0, 0.45, N)
).round(0)
population["premium"] = (rng.random(N) < 0.10 + 0.25 * (population["income"] > 60_000)).astype(int)

MU    = population["income"].mean()
SIGMA = population["income"].std(ddof=0)
P     = population["premium"].mean()

print(f"TRUE population mean income (mu)      : {MU:,.2f}")
print(f"TRUE population sd income  (sigma)    : {SIGMA:,.2f}")
print(f"TRUE premium proportion    (p)        : {P:.4f}")
print()
print(population.groupby("region")["income"].agg(["count", "mean"]).round(0))

---
## 4.2 Probability sampling methods

In **probability sampling** every unit has a known, non-zero chance of selection. That is
what licenses you to generalise. Four workhorses:

### (a) Simple Random Sampling (SRS)
Every unit equally likely; every subset of size $n$ equally likely.
*Pros:* unbiased, simple theory. *Cons:* needs a full list of the population (a *sampling
frame*), and can miss small subgroups entirely.

In [ ]:
n = 500
srs = population.sample(n=n, random_state=1)

print(f"SRS estimate of mean income : {srs['income'].mean():,.2f}   (true {MU:,.2f})")
print(f"Error                       : {srs['income'].mean() - MU:+,.2f}")
print("\nRegion composition:")
print(pd.DataFrame({
    "population %": population["region"].value_counts(normalize=True).round(3),
    "sample %":     srs["region"].value_counts(normalize=True).round(3),
}))

### (b) Systematic sampling
Order the population, pick a random start, then take every $k$-th unit where
$k = N/n$.

*Pros:* trivial to run in the field (every 10th customer through the door).
*Cons:* **dangerous if the list has a periodic pattern** matching $k$ — e.g. sampling every
7th day always lands on a Monday.

In [ ]:
k = N // n
start = rng.integers(0, k)
systematic = population.iloc[start::k]

print(f"Every {k}-th unit starting at index {start} -> n = {len(systematic)}")
print(f"Estimate of mean income : {systematic['income'].mean():,.2f}   (true {MU:,.2f})")

# The failure mode: a population with hidden periodicity
sales = pd.DataFrame({"day": np.arange(700)})
sales["weekday"] = sales["day"] % 7
sales["revenue"] = 1000 + 600 * (sales["weekday"] >= 5) + rng.normal(0, 50, 700)   # weekend spike

every_7th = sales.iloc[0::7]
print(f"\nTrue mean daily revenue        : {sales['revenue'].mean():,.1f}")
print(f"Sampling every 7th day (biased): {every_7th['revenue'].mean():,.1f}")
print("It only ever sees one weekday, so it misses the weekend spike entirely.")

### (c) Stratified sampling
Split the population into homogeneous **strata** (region, age band, device type), then take
a random sample *within* each stratum. Allocate proportionally so the sample mirrors the
population.

*Pros:* **guarantees representation** of every stratum and **reduces variance** — often
substantially. *Cons:* you need to know the strata in advance.

This is exactly what `train_test_split(..., stratify=y)` does in scikit-learn, and why you
should always use it for imbalanced classification.

In [ ]:
def stratified_sample(df, strata_col, n, rng_seed):
    '''Proportional allocation: each stratum contributes its population share of n.'''
    shares = df[strata_col].value_counts(normalize=True)
    parts = []
    for stratum, share in shares.items():
        take = int(round(share * n))
        parts.append(df[df[strata_col] == stratum].sample(n=take, random_state=rng_seed))
    return pd.concat(parts)

strat = stratified_sample(population, "region", n, rng_seed=1)

print(f"Stratified estimate of mean income : {strat['income'].mean():,.2f}   (true {MU:,.2f})")
print("\nRegion composition is matched by construction:")
print(pd.DataFrame({
    "population %": population["region"].value_counts(normalize=True).round(3),
    "stratified %": strat["region"].value_counts(normalize=True).round(3),
}))

In [ ]:
# Does stratification really reduce variance? Repeat both methods 400 times.
reps = 400
srs_means, strat_means = [], []
for i in range(reps):
    srs_means.append(population.sample(n=n, random_state=i)["income"].mean())
    strat_means.append(stratified_sample(population, "region", n, rng_seed=i)["income"].mean())

srs_means, strat_means = np.array(srs_means), np.array(strat_means)

plt.hist(srs_means,   bins=30, alpha=0.6, label=f"SRS (sd={srs_means.std():,.0f})",   color="steelblue")
plt.hist(strat_means, bins=30, alpha=0.6, label=f"Stratified (sd={strat_means.std():,.0f})", color="seagreen")
plt.axvline(MU, color="crimson", lw=2, label="true mean")
plt.xlabel("estimated mean income"); plt.ylabel("frequency")
plt.title("Stratifying on a variable related to the outcome tightens the estimate")
plt.legend(); plt.show()

print(f"SRS        : mean of estimates {srs_means.mean():,.1f}, sd {srs_means.std():,.1f}")
print(f"Stratified : mean of estimates {strat_means.mean():,.1f}, sd {strat_means.std():,.1f}")
print(f"Variance reduction: {(1 - strat_means.var()/srs_means.var())*100:.1f}%")

### (d) Cluster sampling
Divide the population into naturally occurring **clusters** (schools, city blocks, stores),
randomly select whole clusters, and measure **everyone** inside them.

*Pros:* cheap when travel or setup cost dominates — you visit 20 schools, not 400 scattered
households. *Cons:* people within a cluster are similar, so effective sample size is
smaller than $n$ suggests. The inflation factor is the **design effect**.

**Stratified vs. cluster in one line:** stratified takes *some* from *every* group; cluster
takes *all* from *some* groups.

In [ ]:
# 500 clusters of 100 people, with a real cluster effect on income
n_clusters, per_cluster = 500, 100
pop2 = pd.DataFrame({"cluster": np.repeat(np.arange(n_clusters), per_cluster)})
cluster_mean = rng.normal(50_000, 12_000, n_clusters)          # clusters genuinely differ
pop2["income"] = cluster_mean[pop2["cluster"]] + rng.normal(0, 6_000, len(pop2))

true_mean = pop2["income"].mean()

# Cluster sample: 5 whole clusters (n = 500)
chosen = rng.choice(n_clusters, size=5, replace=False)
clus = pop2[pop2["cluster"].isin(chosen)]

# SRS of the same size for comparison
srs2 = pop2.sample(n=500, random_state=3)

print(f"True mean          : {true_mean:,.0f}")
print(f"Cluster sample     : {clus['income'].mean():,.0f}   (n={len(clus)})")
print(f"SRS of same size   : {srs2['income'].mean():,.0f}   (n={len(srs2)})")

# Repeat to compare variability
c_est = [pop2[pop2['cluster'].isin(rng.choice(n_clusters, 5, replace=False))]["income"].mean()
         for _ in range(300)]
s_est = [pop2.sample(500, random_state=i)["income"].mean() for i in range(300)]
print(f"\nsd of cluster estimates : {np.std(c_est):,.0f}")
print(f"sd of SRS estimates     : {np.std(s_est):,.0f}")
print(f"Design effect (variance ratio): {np.var(c_est)/np.var(s_est):.1f}x")
print("Same n, far worse precision -- the cost of clustering.")

---
## 4.3 Non-probability sampling and bias

These are common in practice and **cannot** support valid generalisation, but you should
recognise them by name:

| Method | Description | Bias it creates |
|---|---|---|
| **Convenience** | Whoever is easy to reach | Unknown and unmeasurable |
| **Voluntary response** | People opt in (online polls, reviews) | Strong opinions over-represented |
| **Judgement / purposive** | Expert picks "typical" cases | Researcher's assumptions |
| **Quota** | Fill counts per group, non-randomly within group | Selection bias within quotas |
| **Snowball** | Participants recruit friends | Network homogeneity |

### Three named biases to watch for

- **Selection bias** — the sampling mechanism correlates with the outcome
- **Non-response bias** — the people who decline differ from those who reply
- **Survivorship bias** — you only observe the units that made it (the classic: analysing
  only surviving companies, or planes that returned from missions)

In [ ]:
# Non-response bias: high earners are less likely to disclose their income.
p_respond = np.where(population["income"] > 70_000, 0.25, 0.85)
responded = population[rng.random(N) < p_respond]

print(f"True population mean : {MU:,.0f}")
print(f"Respondents-only mean: {responded['income'].mean():,.0f}")
print(f"Bias                 : {responded['income'].mean() - MU:+,.0f}  "
      f"({(responded['income'].mean()/MU - 1)*100:+.1f}%)")
print(f"Response rate        : {len(responded)/N:.1%}")
print()
print("Note that n is huge (tens of thousands) yet the estimate is badly wrong.")
print("A LARGER biased sample is not a better sample -- it is a more confidently wrong one.")

In [ ]:
# Survivorship bias: judging a strategy by looking only at the survivors.
n_funds, years = 1_000, 10
returns = rng.normal(0.02, 0.20, size=(n_funds, years))     # true mean return only 2%
wealth = np.cumprod(1 + returns, axis=1)

# Funds are closed if they ever fall below 60% of starting value
survived = (wealth > 0.6).all(axis=1)

print(f"All funds     : mean 10-year growth = {wealth[:, -1].mean():.3f}")
print(f"Survivors only: mean 10-year growth = {wealth[survived, -1].mean():.3f}")
print(f"Survivors      : {survived.sum()} of {n_funds} funds ({survived.mean():.1%})")
print("\nLooking only at funds that still exist makes the strategy look far better than it is.")

---
## 4.4 The sampling distribution and the standard error

Take a sample, compute $\bar{x}$. Take another sample, compute $\bar{x}$ again. Repeat
forever. The distribution of those values is the **sampling distribution of the mean**.

Two facts about it:

$$E[\bar{X}] = \mu \qquad \text{(unbiased)}$$

$$\operatorname{SD}(\bar{X}) = \frac{\sigma}{\sqrt{n}} \equiv \textbf{standard error (SE)}$$

The $\sqrt{n}$ is the single most consequential formula in applied statistics:

- To **halve** your uncertainty you must **quadruple** your sample size
- Going from $n=100$ to $n=400$ halves the SE; going to $n=10{,}000$ divides it by 10
- There are steeply diminishing returns to collecting more data

> **Standard deviation vs. standard error.** SD describes the spread of *individual
> observations*. SE describes the spread of a *statistic*. Reporting one when you mean the
> other is the most common error in published charts.

In [ ]:
sizes = [5, 20, 100, 500]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), sharex=True)

incomes = population["income"].to_numpy()

def draw(m):
    '''One simple random sample of size m from the population (as a NumPy array).'''
    return incomes[rng.integers(0, len(incomes), m)]

for ax, m in zip(axes, sizes):
    means = np.array([draw(m).mean() for _ in range(3_000)])
    ax.hist(means, bins=45, color="steelblue", edgecolor="none")
    ax.axvline(MU, color="crimson", lw=2)
    ax.set_title(f"n = {m}\nobserved SE = {means.std():,.0f}\ntheory SE = {SIGMA/np.sqrt(m):,.0f}",
                 fontsize=9)
    ax.set_xlabel("sample mean")
plt.tight_layout(); plt.show()

In [ ]:
ns = np.arange(5, 2001)
se = SIGMA / np.sqrt(ns)

plt.plot(ns, se, lw=2, color="steelblue")
for m in (100, 400, 1600):
    plt.scatter([m], [SIGMA/np.sqrt(m)], color="crimson", zorder=5)
    plt.annotate(f"n={m}\nSE={SIGMA/np.sqrt(m):,.0f}", (m, SIGMA/np.sqrt(m)),
                 textcoords="offset points", xytext=(10, 12), fontsize=8)
plt.xlabel("sample size n"); plt.ylabel("standard error of the mean")
plt.title("Diminishing returns: SE shrinks like 1/sqrt(n)")
plt.show()

---
## 4.5 The Central Limit Theorem

> **Central Limit Theorem.** Let $X_1, \dots, X_n$ be independent and identically
> distributed with mean $\mu$ and **finite** variance $\sigma^2$. Then as $n$ grows,
>
> $$\bar{X}_n \;\xrightarrow{\ d\ }\; N\!\left(\mu,\ \frac{\sigma^2}{n}\right)$$
>
> equivalently $\dfrac{\bar{X}_n - \mu}{\sigma/\sqrt{n}} \to N(0,1)$.

Three things the CLT does **not** say, which students routinely get wrong:

1. ❌ It does not say your *data* becomes Normal. Only the *sample mean* does.
2. ❌ It does not say $n = 30$ is always enough. That rule of thumb fails badly for
   heavily skewed data (see below).
3. ❌ It does not apply when the variance is infinite (e.g. the Cauchy distribution).

Why it matters: because of the CLT, we can build confidence intervals and run t-tests
*without knowing the population distribution*.

In [ ]:
populations = {
    "Uniform":       lambda k: rng.uniform(0, 1, k),
    "Exponential":   lambda k: rng.exponential(1, k),
    "Bimodal":       lambda k: np.where(rng.random(k) < 0.5, rng.normal(-3, 0.6, k), rng.normal(3, 0.6, k)),
    "Heavy-skew":    lambda k: rng.lognormal(0, 1.6, k),
}
sample_sizes = [1, 5, 30]

fig, axes = plt.subplots(len(populations), len(sample_sizes),
                         figsize=(13, 3.0 * len(populations)))
for i, (name, gen) in enumerate(populations.items()):
    for j, m in enumerate(sample_sizes):
        means = gen(6_000 * m).reshape(6_000, m).mean(axis=1)
        z = (means - means.mean()) / means.std()
        ax = axes[i, j]
        ax.hist(z, bins=60, density=True, color="steelblue", edgecolor="none")
        xs = np.linspace(-4, 4, 200)
        ax.plot(xs, stats.norm.pdf(xs), color="crimson", lw=1.6)
        ax.set_xlim(-4, 4); ax.set_yticks([])
        ax.set_title(f"{name}, n={m}  (skew {stats.skew(means):+.2f})", fontsize=9)
plt.tight_layout(); plt.show()

print("Red curve = standard Normal. Read each row left to right: the histogram")
print("straightens out as n grows. The heavy-skew row is still not there at n=30 --")
print("that is why 'n > 30 is enough' is a rule of thumb, not a theorem.")

In [ ]:
# The CLT needs FINITE variance. Cauchy has none, and the mean never converges.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for m, colour in [(10, "steelblue"), (1000, "seagreen")]:
    means = rng.standard_cauchy((4_000, m)).mean(axis=1)
    ax[0].hist(np.clip(means, -20, 20), bins=80, alpha=0.6, density=True,
               label=f"n={m}", color=colour)
ax[0].set_title("Cauchy: sample mean does NOT stabilise")
ax[0].legend()

running = np.cumsum(rng.standard_cauchy(50_000)) / np.arange(1, 50_001)
ax[1].plot(running, lw=0.8, color="crimson")
ax[1].axhline(0, ls="--", color="black")
ax[1].set_xscale("log"); ax[1].set_title("Running mean of Cauchy draws: never settles")
ax[1].set_xlabel("number of observations")
plt.tight_layout(); plt.show()

---
## 4.6 Confidence intervals

A **confidence interval** turns a point estimate into a range that reflects sampling
uncertainty. For a mean with unknown $\sigma$:

$$\bar{x} \pm t_{\alpha/2,\ n-1} \cdot \frac{s}{\sqrt{n}}$$

For a proportion:

$$\hat{p} \pm z_{\alpha/2} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

### What "95% confidence" actually means

✅ **Correct:** *"If I repeated this whole sampling procedure many times, 95% of the
intervals I construct would contain the true parameter."* The randomness is in the
interval, not the parameter.

❌ **Wrong:** *"There is a 95% probability that $\mu$ is in this interval."* $\mu$ is a
fixed number — it either is or is not in your particular interval.

Let's *see* the correct interpretation by building 100 intervals.

In [ ]:
def ci_mean(sample, conf=0.95):
    m  = len(sample)
    xb = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(m)
    t  = stats.t(m - 1).ppf(1 - (1 - conf) / 2)
    return xb, xb - t * se, xb + t * se

one = draw(200)
xb, lo, hi = ci_mean(one)
print(f"Sample mean          : {xb:,.0f}")
print(f"95% CI               : [{lo:,.0f}, {hi:,.0f}]")
print(f"True mu              : {MU:,.0f}  -> {'inside' if lo <= MU <= hi else 'OUTSIDE'}")
print(f"Margin of error      : +/-{(hi-lo)/2:,.0f}")

In [ ]:
# Draw 100 independent samples and their 95% CIs
trials = 100
fig, ax = plt.subplots(figsize=(9, 6))
covered = 0
for i in range(trials):
    s = draw(200)
    xb, lo, hi = ci_mean(s)
    ok = lo <= MU <= hi
    covered += ok
    ax.plot([lo, hi], [i, i], color="steelblue" if ok else "crimson", lw=1.6)
    ax.plot(xb, i, "o", ms=2.5, color="black")

ax.axvline(MU, color="darkgreen", lw=2, label="true mu")
ax.set_xlabel("income"); ax.set_ylabel("sample number")
ax.set_title(f"100 independent 95% CIs: {covered} contained mu (red ones missed)")
ax.legend()
plt.tight_layout(); plt.show()

print(f"Empirical coverage: {covered}/{trials} = {covered/trials:.0%}   (nominal 95%)")

In [ ]:
# Confidence interval for a proportion
sample = population.sample(1_000, random_state=5)
p_hat = sample["premium"].mean()
m = len(sample)

se = np.sqrt(p_hat * (1 - p_hat) / m)
z = stats.norm.ppf(0.975)
print(f"p_hat        = {p_hat:.4f}")
print(f"SE           = {se:.4f}")
print(f"95% CI       = [{p_hat - z*se:.4f}, {p_hat + z*se:.4f}]")
print(f"True p       = {P:.4f}")

# scipy's exact (Clopper-Pearson) interval is safer for small n or extreme p
exact = stats.binomtest(int(sample['premium'].sum()), m).proportion_ci(method="exact")
print(f"\nExact 95% CI = [{exact.low:.4f}, {exact.high:.4f}]")

# Wider confidence = wider interval
for conf in (0.80, 0.90, 0.95, 0.99):
    zz = stats.norm.ppf(1 - (1-conf)/2)
    print(f"{conf:.0%} CI width: {2*zz*se:.4f}")

---
## 4.7 The bootstrap: confidence intervals with no formula

What if you want a CI for the **median**, or the 90th percentile, or the ratio of two
statistics — quantities with no clean formula? The **bootstrap** solves this by resampling
your own data *with replacement*:

1. Draw a resample of size $n$ from your sample, **with replacement**
2. Compute the statistic
3. Repeat ~10,000 times
4. The 2.5th and 97.5th percentiles of those values form a 95% CI

The idea (Efron, 1979) is that your sample stands in for the population. It is one of the
most practically useful ideas in modern statistics and it costs three lines of code.

In [ ]:
def bootstrap_ci(sample, statistic=np.mean, B=10_000, conf=0.95, seed=0):
    '''Percentile bootstrap confidence interval for any statistic.'''
    r = np.random.default_rng(seed)
    m = len(sample)
    boot = np.array([statistic(r.choice(sample, m, replace=True)) for _ in range(B)])
    a = (1 - conf) / 2
    return boot, np.quantile(boot, [a, 1 - a])

s = draw(300)

boot_mean, ci_m = bootstrap_ci(s, np.mean)
boot_med,  ci_d = bootstrap_ci(s, np.median)

print(f"MEAN   : estimate {s.mean():,.0f}   bootstrap 95% CI [{ci_m[0]:,.0f}, {ci_m[1]:,.0f}]")
print(f"         formula-based CI            [{ci_mean(s)[1]:,.0f}, {ci_mean(s)[2]:,.0f}]")
print(f"MEDIAN : estimate {np.median(s):,.0f}   bootstrap 95% CI [{ci_d[0]:,.0f}, {ci_d[1]:,.0f}]")
print(f"         (no simple textbook formula exists for this)")
print(f"\nTrue population mean {MU:,.0f}, true median {np.median(incomes):,.0f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, boot, ci, name, truth in [
    (ax[0], boot_mean, ci_m, "mean",   MU),
    (ax[1], boot_med,  ci_d, "median", np.median(incomes)),
]:
    a.hist(boot, bins=60, color="steelblue", edgecolor="none")
    a.axvline(ci[0], color="crimson", ls="--", lw=2)
    a.axvline(ci[1], color="crimson", ls="--", lw=2)
    a.axvline(truth, color="darkgreen", lw=2, label="true value")
    a.set_title(f"Bootstrap distribution of the {name}")
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# scipy has a built-in bootstrap with better (BCa) intervals
res = stats.bootstrap((s,), np.median, confidence_level=0.95,
                      n_resamples=5_000, random_state=1, method="BCa")
print(f"scipy BCa bootstrap CI for the median: "
      f"[{res.confidence_interval.low:,.0f}, {res.confidence_interval.high:,.0f}]")
print(f"Bootstrap SE of the median           : {res.standard_error:,.0f}")

---
## 4.8 How large a sample do I need?

Rearrange the margin-of-error formula. For a **mean** with target margin $E$:

$$n = \left(\frac{z_{\alpha/2}\,\sigma}{E}\right)^2$$

For a **proportion**:

$$n = \frac{z_{\alpha/2}^2\, p(1-p)}{E^2}$$

Since $p(1-p)$ peaks at $p = 0.5$, using $p = 0.5$ gives the **conservative** (largest)
sample size when you have no prior estimate. That is why national polls so often report
about 1,000 respondents: $n = (1.96^2 \times 0.25)/0.03^2 \approx 1{,}068$ for a ±3%
margin.

In [ ]:
def n_for_mean(sigma, margin, conf=0.95):
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    return int(np.ceil((z * sigma / margin) ** 2))

def n_for_proportion(margin, p=0.5, conf=0.95):
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    return int(np.ceil(z**2 * p * (1 - p) / margin**2))

print("Sample size for estimating a MEAN (sigma from the population):")
for margin in (2000, 1000, 500, 250):
    print(f"  margin +/-{margin:>5,}: n = {n_for_mean(SIGMA, margin):>7,}")

print("\nSample size for estimating a PROPORTION (worst case p=0.5):")
for margin in (0.05, 0.03, 0.02, 0.01):
    print(f"  margin +/-{margin:.0%}: n = {n_for_proportion(margin):>7,}")

print("\nIf you already believe p is near 0.10, you need far fewer:")
for margin in (0.05, 0.03, 0.02, 0.01):
    print(f"  margin +/-{margin:.0%}: n = {n_for_proportion(margin, p=0.10):>7,}")

In [ ]:
# Verify empirically that the computed n really delivers the promised margin
target_margin = 1000
n_needed = n_for_mean(SIGMA, target_margin)
print(f"Formula says n = {n_needed:,} for a +/-{target_margin:,} margin")

widths = []
for i in range(400):
    s = draw(n_needed)
    _, lo, hi = ci_mean(s)
    widths.append((hi - lo) / 2)
print(f"Average achieved margin over 400 samples: +/-{np.mean(widths):,.0f}")

---
## 4.9 Sampling in machine learning

Every ML workflow is built on sampling decisions. The vocabulary maps directly:

| ML practice | Sampling concept |
|---|---|
| Train/test split | Simple random sampling |
| `stratify=y` | Stratified sampling |
| k-fold cross-validation | Systematic partitioning |
| Bagging / random forests | Bootstrap resampling |
| Mini-batch SGD | Repeated random sampling |
| **Data leakage from a bad split** | Violating independence |

The rule that matters most: **split before you look**. Any decision made after seeing the
test set (feature choices, hyperparameters, even "let me just check the distribution")
leaks information and inflates your reported score.

In [ ]:
from sklearn.model_selection import train_test_split

X = population[["income"]].to_numpy()
y = population["premium"].to_numpy()

# Naive split vs stratified split on an imbalanced target
_, _, y_tr_a, y_te_a = train_test_split(X, y, test_size=0.2, random_state=0)
_, _, y_tr_b, y_te_b = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

print(f"Overall positive rate       : {y.mean():.4f}")
print(f"Random split   test rate    : {y_te_a.mean():.4f}")
print(f"Stratified     test rate    : {y_te_b.mean():.4f}")

# With a small dataset the difference becomes serious
small_idx = rng.choice(N, 200, replace=False)
Xs, ys = X[small_idx], y[small_idx]
rates_random, rates_strat = [], []
for i in range(300):
    _, _, _, t1 = train_test_split(Xs, ys, test_size=0.25, random_state=i)
    _, _, _, t2 = train_test_split(Xs, ys, test_size=0.25, random_state=i, stratify=ys)
    rates_random.append(t1.mean()); rates_strat.append(t2.mean())

print(f"\nn=200, 300 repeats -- sd of the test-set positive rate:")
print(f"  random     : {np.std(rates_random):.4f}")
print(f"  stratified : {np.std(rates_strat):.4f}   <- much more stable evaluation")

---
## Exercises

**Exercise 1.** A university wants to estimate average study hours. Classify each plan as a
valid probability sampling method or a biased approach, and name the specific risk:
(a) email every student, use whoever replies;
(b) randomly pick 20 of 400 tutorial groups and survey everyone in them;
(c) sort all students by ID and take every 25th;
(d) sample 100 students from each year group in proportion to year size;
(e) survey students in the library on a Wednesday evening.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
answers = pd.DataFrame([
    ("(a) whoever replies to an email", "Voluntary response",  "NOT probability",
     "Non-response bias: keen/anxious students over-represented"),
    ("(b) 20 of 400 tutorial groups",   "Cluster",             "Valid probability",
     "Design effect: groups are internally similar, so effective n is smaller"),
    ("(c) every 25th student by ID",    "Systematic",          "Valid probability",
     "Risk only if IDs encode something (e.g. course or intake year)"),
    ("(d) proportional by year group",  "Stratified",          "Valid probability",
     "Best precision here; needs accurate year-group sizes"),
    ("(e) library on Wednesday night",  "Convenience",         "NOT probability",
     "Selection bias: library users study far more than average"),
], columns=["plan", "method", "validity", "main risk"])

for _, r in answers.iterrows():
    print(f"{r['plan']}\n   method   : {r['method']}\n   validity : {r['validity']}"
          f"\n   risk     : {r['main risk']}\n")

**Exercise 2.** A sample of 64 batteries has mean life 210 hours and sample sd 24 hours.
(a) Compute the standard error.
(b) Build a 95% confidence interval.
(c) How many batteries would you need to cut the margin of error in half?
(d) Recompute the CI at 99% confidence and explain what changed.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
m, xbar, s = 64, 210, 24

se = s / np.sqrt(m)
t95 = stats.t(m - 1).ppf(0.975)
t99 = stats.t(m - 1).ppf(0.995)

print(f"(a) SE = 24/sqrt(64) = {se:.3f} hours")
print(f"(b) 95% CI = {xbar:.1f} +/- {t95*se:.2f} = [{xbar - t95*se:.2f}, {xbar + t95*se:.2f}]")
print(f"(c) Halving the margin needs 4x the data: n = {m*4}")
print(f"    check: SE at n=256 is {s/np.sqrt(256):.3f}, exactly half of {se:.3f}")
print(f"(d) 99% CI = {xbar:.1f} +/- {t99*se:.2f} = [{xbar - t99*se:.2f}, {xbar + t99*se:.2f}]")
print("    More confidence requires a wider net -- precision and certainty trade off.")

**Exercise 3.** Demonstrate the CLT yourself on a distribution not used above: the
**Bernoulli(0.05)** (a rare event). Plot the standardised sampling distribution of the mean
for $n = 10, 50, 200, 1000$. At what $n$ does it look Normal? Relate your answer to the
common rule $np \ge 10$ and $n(1-p) \ge 10$.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
p = 0.05
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, m in zip(axes, [10, 50, 200, 1000]):
    means = rng.binomial(1, p, size=(8_000, m)).mean(axis=1)
    z = (means - p) / np.sqrt(p * (1 - p) / m)
    ax.hist(z, bins=50, density=True, color="steelblue", edgecolor="none")
    xs = np.linspace(-4, 4, 200)
    ax.plot(xs, stats.norm.pdf(xs), color="crimson", lw=1.6)
    ax.set_xlim(-4, 4); ax.set_yticks([])
    ax.set_title(f"n={m}   np={m*p:.0f}\nskew={stats.skew(means):+.2f}", fontsize=9)
plt.tight_layout(); plt.show()

print("np >= 10 requires n >= 200 when p = 0.05, and that is exactly where the")
print("histogram starts to look Normal. For rare events you need a LOT of data.")

**Exercise 4 (challenge).** Your company's A/B test dashboard reports a 3.2% conversion
rate from 500 visitors. Product management wants to know whether the true rate could
plausibly be above 4%.
(a) Give a 95% CI using the Normal approximation and using the exact method — do they
agree? Why or why not?
(b) Use a bootstrap to confirm.
(c) How many visitors would you need to distinguish 3.2% from 4.0%?

In [ ]:
# --- Solution 4 -------------------------------------------------------------
n_vis, conv = 500, 16          # 16/500 = 3.2%
p_hat = conv / n_vis

# (a) Normal approximation
se = np.sqrt(p_hat * (1 - p_hat) / n_vis)
z = stats.norm.ppf(0.975)
print(f"(a) Normal-approx 95% CI : [{p_hat - z*se:.4f}, {p_hat + z*se:.4f}]")
ex = stats.binomtest(conv, n_vis).proportion_ci(method="exact")
print(f"    Exact (Clopper-Pearson): [{ex.low:.4f}, {ex.high:.4f}]")
print(f"    np_hat = {n_vis*p_hat:.0f} -- just above the usual threshold of 10, so the")
print("    approximation is usable but the exact interval is slightly asymmetric.")
print(f"    4% IS inside the interval -> we cannot rule it out.")

# (b) Bootstrap
obs = np.concatenate([np.ones(conv), np.zeros(n_vis - conv)])
_, ci_b = bootstrap_ci(obs, np.mean, B=10_000, seed=2)
print(f"\n(b) Bootstrap 95% CI     : [{ci_b[0]:.4f}, {ci_b[1]:.4f}]")

# (c) Sample size to resolve a 0.8 percentage-point difference
margin = 0.004                                   # half of the 0.8pp gap
n_req = n_for_proportion(margin, p=0.036)
print(f"\n(c) To resolve a 0.8pp gap you need about n = {n_req:,} visitors per arm.")
print("    Small effects on small baselines are expensive to measure -- this is the")
print("    single most common reason A/B tests are declared 'inconclusive'.")

---
## Summary

| Concept | Key point |
|---|---|
| Parameter vs statistic | $\mu$ is fixed and unknown; $\bar{x}$ is known and random |
| Simple random sampling | Unbiased baseline; needs a sampling frame |
| Stratified sampling | Some from every group → lower variance |
| Cluster sampling | All from some groups → cheaper, less precise |
| Systematic sampling | Every $k$-th; beware periodicity |
| Standard error | $\operatorname{SE} = \sigma/\sqrt{n}$ — quadruple $n$ to halve it |
| CLT | $\bar{X} \approx N(\mu, \sigma^2/n)$ for large $n$, finite variance |
| Confidence interval | $\bar{x} \pm t \cdot s/\sqrt{n}$; 95% refers to the *procedure* |
| Bootstrap | Resample with replacement; works for any statistic |
| Sample size | $n = (z\sigma/E)^2$ for means, $z^2p(1-p)/E^2$ for proportions |
| Bias | A bigger biased sample is worse, not better |

**Next up:** [Notebook 5 — Correlation and Covariance](5.%20Correlation%20and%20Covariance.ipynb),
where we move from one variable to relationships between two.